# SECAI — Data Cleaning
Source: `timesheet_output.xlsx` → sheet `All Records`

Each cell tackles one specific issue found during initial screening.

In [ ]:
import re
import pandas as pd

FILE_PATH  = "timesheet_output.xlsx"
SHEET_NAME = "All Records"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, dtype=str)
df.columns = df.columns.str.strip()
for col in df.columns:
    df[col] = df[col].str.strip()

print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")

## Fix 1 · Translate Spanish day names to English

In [ ]:
SPANISH_MAP = {
    "LUNES":      "Monday",
    "MARTES":     "Tuesday",
    "MIERCOLES":  "Wednesday",
    "MIÉRCOLES":  "Wednesday",
    "JUEVES":     "Thursday",
    "VIERNES":    "Friday",
    "SABADO":     "Saturday",
    "SÁBADO":     "Saturday",
    "DOMINGO":    "Sunday",
}

before = df["Day"].value_counts(dropna=False).to_dict()

df["Day"] = df["Day"].apply(
    lambda v: SPANISH_MAP.get(str(v).upper().strip(), v) if pd.notna(v) else v
)

after = df["Day"].value_counts(dropna=False)

# Summary of what changed
changed = {k: v for k, v in before.items() if str(k).upper().strip() in SPANISH_MAP}
print(f"Translated {sum(changed.values())} rows across {len(changed)} Spanish value(s):")
for spanish, count in changed.items():
    english = SPANISH_MAP[str(spanish).upper().strip()]
    print(f"  '{spanish}' ({count} rows)  →  '{english}'")

print("\nDistinct Day values after fix:")
display(after.rename_axis("Day").reset_index(name="Count"))

## Fix 1 · Check — rows where Day is still not a standard weekday
After the Spanish translation, any remaining non-standard value needs manual review.

In [ ]:
VALID_DAYS = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"}

mask = ~df["Day"].apply(lambda v: str(v).strip().title() in VALID_DAYS if pd.notna(v) else False)
odd_days = df[mask][["Source File", "Page", "Date", "Day", "Employee Name"]].reset_index(drop=True)

print(f"{len(odd_days)} rows with non-standard Day values. Showing up to 10:")
display(odd_days.head(10))